# Tensor Export Validation Notebook (`tensor-inspection-v0.2`)

This notebook performs **inspection and validation only** for one exported `.npz` tensor file.
It does **not** patch files, rewrite datasets, or launch training.

Expected final verdict:

- `READY_TO_PROCEED`
- or `PATCH_REQUIRED`

In [ ]:
# Imports
from pathlib import Path
import json
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

plt.rcParams["figure.figsize"] = (12, 4)

## 1. Configuration / file selection section

Select one `.npz` file (via widget) and specify manifest paths.

In [ ]:
# Section 1 - interactive file selection widgets


def detect_repo_root(start: Path) -> Path:
    """Walk up until a folder containing `preprocessing` is found."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "preprocessing").exists():
            return candidate
    return start


PROJECT_ROOT = detect_repo_root(Path.cwd())
DEFAULT_NPZ_SEARCH_ROOTS = [
    PROJECT_ROOT / "preprocessing" / "03.training-export" / "output",
    PROJECT_ROOT / "preprocessing",
]


def discover_npz_files(search_roots=None, limit=5000):
    if search_roots is None:
        search_roots = DEFAULT_NPZ_SEARCH_ROOTS

    found = []
    seen = set()
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("*.npz"):
            if p.is_file():
                rp = p.resolve()
                if rp not in seen:
                    seen.add(rp)
                    found.append(rp)
                if len(found) >= limit:
                    break
        if len(found) >= limit:
            break
    return sorted(found)


def discover_manifest(default_kind="windows"):
    manifests_root = PROJECT_ROOT / "preprocessing" / "03.training-export" / "output"
    if not manifests_root.exists():
        return ""
    patt = "*_windows.parquet" if default_kind == "windows" else "*_files.parquet"
    candidates = sorted(manifests_root.rglob(patt))
    return str(candidates[-1]) if candidates else ""


npz_path_filter = widgets.Text(
    value="",
    description="Path filter:",
    placeholder="optional substring, e.g. -6db-pump/id_00",
    layout=widgets.Layout(width="95%"),
)

npz_select = widgets.Select(
    options=[],
    value=None,
    rows=14,
    description=".npz files",
    layout=widgets.Layout(width="95%"),
)

npz_path_text = widgets.Text(
    value="",
    description="npz_path",
    placeholder="selected .npz path appears here",
    layout=widgets.Layout(width="95%"),
)

window_manifest_path_text = widgets.Text(
    value=discover_manifest("windows"),
    description="window_manifest_path",
    layout=widgets.Layout(width="95%"),
)

clip_manifest_path_text = widgets.Text(
    value=discover_manifest("files"),
    description="clip_manifest_path",
    placeholder="Optional",
    layout=widgets.Layout(width="95%"),
)

refresh_button = widgets.Button(description="Refresh file list", button_style="")
load_button = widgets.Button(description="Use selected .npz", button_style="primary")
status_output = widgets.Output()


def refresh_npz_options(*_):
    paths = discover_npz_files(DEFAULT_NPZ_SEARCH_ROOTS)
    filter_text = npz_path_filter.value.strip().lower()

    if filter_text:
        paths = [p for p in paths if filter_text in str(p).lower()]

    options = []
    for p in paths:
        try:
            label = str(p.relative_to(PROJECT_ROOT))
        except Exception:
            label = str(p)
        options.append((label, str(p)))

    npz_select.options = options
    npz_select.value = options[0][1] if options else None

    # Keep text in sync with current selection.
    npz_path_text.value = npz_select.value or ""

    with status_output:
        clear_output(wait=True)
        print(f"Repo root: {PROJECT_ROOT}")
        print(f"Discovered {len(options)} matching .npz files.")


def on_load_clicked(*_):
    with status_output:
        clear_output(wait=True)
        if not npz_select.value:
            print("No .npz file selected.")
            return

        npz_path_text.value = npz_select.value
        print("Selected .npz path copied to npz_path field.")
        print(npz_path_text.value)


def on_select_change(change):
    if change.get("name") == "value":
        npz_path_text.value = change.get("new") or ""


npz_select.observe(on_select_change, names="value")
refresh_button.on_click(refresh_npz_options)
load_button.on_click(on_load_clicked)

controls_row = widgets.HBox([refresh_button, load_button])

display(Markdown("### Select one tensor export (`.npz`)"))
display(npz_path_filter)
display(controls_row)
display(npz_select)
display(status_output)
display(Markdown("### File inputs"))
display(npz_path_text)
display(window_manifest_path_text)
display(clip_manifest_path_text)

refresh_npz_options()


## 2-13. Full validation pass (one run)

Use the cell below to run all required sections in one execution.

## 2. NPZ inventory section

## 3. Core shapes and dtypes validation

## 4. Fixed configuration value checks

## 5. Window count and frame-start checks

## 6. Value range checks

## 7. Internal consistency checks

## 8. Sparsity / occupancy statistics

## 9. Continuous-versus-quantized agreement

## 10. Visual inspection section

## 11. Manifest alignment checks

## 12. Source identity / uniqueness checks

## 13. Final verdict section

These sections are executed in order by the full validation pass cell below.


In [ ]:
# Sections 2-13 implemented as one full validation pass
SMALL_TOL = 1e-6
LAST_RUN = {}


def _normalize_path_str(x):
    return str(x).replace("\\", "/")


def _as_path_or_none(txt):
    if txt is None:
        return None
    s = str(txt).strip()
    if not s:
        return None
    p = Path(s)
    if not p.is_absolute():
        p = (PROJECT_ROOT / p).resolve()
    return p


def _has_array(npz, key):
    return key in npz.files


def _get_scalar_int(npz, key):
    if key not in npz.files:
        return None, False, "missing"
    arr = npz[key]
    if np.asarray(arr).ndim != 0:
        return None, False, f"not scalar (shape={arr.shape})"
    try:
        val = int(np.asarray(arr).item())
    except Exception as exc:
        return None, False, f"cannot cast to int: {exc}"
    return val, True, "ok"


def _print_h2(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def _record(checks, section, check, passed, detail=""):
    checks.append({
        "section": section,
        "check": check,
        "passed": bool(passed),
        "detail": str(detail),
    })
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check} :: {detail}")


def _safe_min_max(arr):
    if arr is None or arr.size == 0:
        return None, None
    return np.min(arr), np.max(arr)


def _show_visuals(normalized_window, height_bins, active_mask, indices):
    nrows = len(indices)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(15, 3.5 * nrows), squeeze=False)
    columns = [
        ("normalized_window", normalized_window, "viridis"),
        ("height_bins", height_bins, "magma"),
        ("active_mask", active_mask.astype(float), "gray"),
    ]
    for r, idx in enumerate(indices):
        for c, (name, arr, cmap) in enumerate(columns):
            ax = axes[r][c]
            im = ax.imshow(arr[idx], aspect="auto", origin="lower", cmap=cmap)
            ax.set_title(f"{name}[{idx}]")
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()


def _find_matching_rows(df, npz_path):
    if df is None or df.empty:
        return df.iloc[0:0], "none"

    npz_abs = _normalize_path_str(Path(npz_path).resolve())
    npz_name = Path(npz_path).name

    rel_candidates = []
    try:
        rel_candidates.append(_normalize_path_str(Path(npz_path).resolve().relative_to(PROJECT_ROOT.resolve())))
    except Exception:
        pass
    rel_candidates.append(_normalize_path_str(Path(npz_path)))

    cols = [
        c for c in df.columns
        if any(tok in c.lower() for tok in ["tensor", "npz", "path", "file"])
    ]

    if not cols:
        return df.iloc[0:0], "no_path_columns"

    best_rows = df.iloc[0:0]
    best_method = "none"

    for col in cols:
        s = df[col].astype(str).map(_normalize_path_str)

        exact_abs = s == npz_abs
        if exact_abs.any():
            rows = df[exact_abs]
            if len(rows) > len(best_rows):
                best_rows = rows
                best_method = f"exact_abs:{col}"

        for rel in rel_candidates:
            exact_rel = s == rel
            if exact_rel.any():
                rows = df[exact_rel]
                if len(rows) > len(best_rows):
                    best_rows = rows
                    best_method = f"exact_rel:{col}"

        name_match = (s == npz_name) | s.str.endswith("/" + npz_name)
        if name_match.any() and len(best_rows) == 0:
            rows = df[name_match]
            if len(rows) > len(best_rows):
                best_rows = rows
                best_method = f"filename:{col}"

    return best_rows, best_method


def run_full_validation(npz_path_text_val, window_manifest_text_val, clip_manifest_text_val=None):
    checks = []
    context = {
        "npz_path": None,
        "npz": None,
        "arrays": {},
        "N": None,
        "window_rows": None,
        "clip_rows": None,
    }

    npz_path = _as_path_or_none(npz_path_text_val)
    window_manifest_path = _as_path_or_none(window_manifest_text_val)
    clip_manifest_path = _as_path_or_none(clip_manifest_text_val)

    context["npz_path"] = npz_path

    _print_h2("2. NPZ inventory section")
    if npz_path is None:
        _record(checks, 2, "npz_path provided", False, "No npz path selected")
        return checks, context
    if not npz_path.exists():
        _record(checks, 2, "npz_path exists", False, f"File not found: {npz_path}")
        return checks, context

    print(f"Selected NPZ: {npz_path}")
    try:
        npz = np.load(npz_path, allow_pickle=False)
        context["npz"] = npz
        keys = list(npz.files)
        _record(checks, 2, "NPZ loaded", True, f"keys={len(keys)}")
        print("Keys:", keys)
        inventory_rows = []
        for k in keys:
            arr = npz[k]
            inventory_rows.append({
                "key": k,
                "shape": tuple(arr.shape),
                "dtype": str(arr.dtype),
                "ndim": int(arr.ndim),
                "size": int(arr.size),
            })
        inv_df = pd.DataFrame(inventory_rows)
        display(inv_df)
    except Exception as exc:
        _record(checks, 2, "NPZ loaded", False, f"load error: {exc}")
        return checks, context

    required_keys = [
        "height_bins",
        "active_mask",
        "normalized_window",
        "frame_starts",
        "window_frames",
        "stride_frames",
        "num_bands",
        "target_sample_rate",
    ]

    arrays = {k: npz[k] for k in npz.files}
    context["arrays"] = arrays

    _print_h2("3. Core shapes and dtypes validation")
    for k in required_keys:
        _record(checks, 3, f"required key exists: {k}", _has_array(npz, k), "present" if _has_array(npz, k) else "missing")

    hb = arrays.get("height_bins")
    am = arrays.get("active_mask")
    nw = arrays.get("normalized_window")
    fs = arrays.get("frame_starts")

    N = None
    if hb is not None and hb.ndim >= 1:
        N = hb.shape[0]
    elif am is not None and am.ndim >= 1:
        N = am.shape[0]
    elif nw is not None and nw.ndim >= 1:
        N = nw.shape[0]
    elif fs is not None and fs.ndim == 1:
        N = fs.shape[0]
    context["N"] = N

    if N is not None:
        _record(checks, 3, "inferred N", True, f"N={N}")
    else:
        _record(checks, 3, "inferred N", False, "Unable to infer N")

    if hb is not None and N is not None:
        _record(checks, 3, "height_bins shape == (N,96,64)", hb.shape == (N, 96, 64), f"shape={hb.shape}")
        _record(checks, 3, "height_bins dtype == uint8", hb.dtype == np.dtype("uint8"), f"dtype={hb.dtype}")
    if am is not None and N is not None:
        _record(checks, 3, "active_mask shape == (N,96,64)", am.shape == (N, 96, 64), f"shape={am.shape}")
        _record(checks, 3, "active_mask dtype == bool", am.dtype == np.dtype(bool), f"dtype={am.dtype}")
    if nw is not None and N is not None:
        _record(checks, 3, "normalized_window shape == (N,96,64)", nw.shape == (N, 96, 64), f"shape={nw.shape}")
        _record(checks, 3, "normalized_window dtype == float32", nw.dtype == np.dtype("float32"), f"dtype={nw.dtype}")

    if fs is not None and N is not None:
        _record(checks, 3, "frame_starts shape == (N,)", fs.shape == (N,), f"shape={fs.shape}")

    for scalar_k in ["window_frames", "stride_frames", "num_bands", "target_sample_rate"]:
        val, ok, msg = _get_scalar_int(npz, scalar_k)
        _record(checks, 3, f"{scalar_k} is scalar int", ok, f"{msg}; value={val}")

    _print_h2("4. Fixed configuration value checks")
    expected = {
        "num_bands": 96,
        "window_frames": 64,
        "stride_frames": 32,
        "target_sample_rate": 16000,
    }
    scalar_values = {}
    for k, exp in expected.items():
        val, ok, msg = _get_scalar_int(npz, k)
        scalar_values[k] = val
        _record(checks, 4, f"{k} == {exp}", ok and (val == exp), f"value={val}; {msg}")

    _print_h2("5. Window count and frame-start checks")
    if all(x is not None for x in [hb, am, nw, fs]):
        same_n = (hb.shape[0] == am.shape[0] == nw.shape[0] == len(fs))
        _record(checks, 5, "N matches across height_bins/active_mask/normalized_window/frame_starts", same_n,
                f"hb={hb.shape[0]}, am={am.shape[0]}, nw={nw.shape[0]}, fs={len(fs)}")

        if len(fs) > 0:
            _record(checks, 5, "first frame_start is 0", int(fs[0]) == 0, f"first={int(fs[0])}")
            diffs = np.diff(fs)
            all_stride_32 = np.all(diffs == 32) if len(diffs) else True
            _record(checks, 5, "np.diff(frame_starts) all 32", bool(all_stride_32), f"unique_diffs={np.unique(diffs) if len(diffs) else []}")
            print("first frame starts:", fs[: min(8, len(fs))])
            print("last frame starts:", fs[max(0, len(fs)-8):])
            print("total windows (N):", len(fs))
            if len(fs) == 18:
                print("10-second heuristic check: N == 18 (matches expected for ~10s clip)")
            else:
                print(f"10-second heuristic check: N = {len(fs)} (not 18; reported only)")

    else:
        _record(checks, 5, "window/frame-start checks executable", False, "Missing one or more required arrays")

    _print_h2("6. Value range checks")
    if hb is not None:
        hb_min, hb_max = _safe_min_max(hb)
        _record(checks, 6, "height_bins.min() >= 0", hb_min is not None and hb_min >= 0, f"min={hb_min}")
        _record(checks, 6, "height_bins.max() <= 23", hb_max is not None and hb_max <= 23, f"max={hb_max}")

    if nw is not None:
        nw_min, nw_max = _safe_min_max(nw)
        _record(checks, 6, "normalized_window.min() >= 0.0", nw_min is not None and nw_min >= 0.0, f"min={nw_min}")
        _record(checks, 6, "normalized_window.max() <= 1.0 + small_tolerance", nw_max is not None and nw_max <= 1.0 + SMALL_TOL, f"max={nw_max}; tol={SMALL_TOL}")

    if am is not None:
        if am.dtype == np.dtype(bool):
            _record(checks, 6, "active_mask contains only boolean values", True, "dtype is bool")
        else:
            uniq = np.unique(am)
            is_booleanish = set(uniq.tolist()).issubset({0, 1, False, True})
            _record(checks, 6, "active_mask contains only boolean values", is_booleanish, f"dtype={am.dtype}; unique_sample={uniq[:8]}")

    _print_h2("7. Internal consistency checks")
    if all(x is not None for x in [hb, am, nw]):
        inactive = ~am.astype(bool)
        active = am.astype(bool)

        inactive_nw_zero = np.all(nw[inactive] == 0)
        inactive_nw_gt0_count = int(np.sum(nw[inactive] > 0))
        inactive_hb_zero = np.all(hb[inactive] == 0)
        inactive_hb_gt0_count = int(np.sum(hb[inactive] > 0))

        active_nw_gt0_count = int(np.sum(nw[active] > 0))
        active_nw_eq0_count = int(np.sum(nw[active] == 0))

        exact_alignment = np.array_equal(hb == 0, ~active)

        _record(checks, 7, "all inactive-mask cells have normalized_window == 0", bool(inactive_nw_zero), f"violations={inactive_nw_gt0_count}")
        _record(checks, 7, "no inactive-mask cells have normalized_window > 0", inactive_nw_gt0_count == 0, f"count={inactive_nw_gt0_count}")
        _record(checks, 7, "all inactive-mask cells have height_bins == 0", bool(inactive_hb_zero), f"violations={inactive_hb_gt0_count}")
        _record(checks, 7, "no inactive-mask cells have height_bins > 0", inactive_hb_gt0_count == 0, f"count={inactive_hb_gt0_count}")
        _record(checks, 7, "active cells with normalized_window > 0", active_nw_gt0_count > 0, f"count={active_nw_gt0_count}")
        _record(checks, 7, "active cells with normalized_window == 0", True, f"count={active_nw_eq0_count}")
        _record(checks, 7, "exact alignment: (height_bins == 0) == (active_mask == False)", bool(exact_alignment), "global equality check")

        if hb.shape[0] > 0:
            w0_active = int(np.sum(active[0]))
            w0_total = int(active[0].size)
            w0_inactive = w0_total - w0_active
            w0_active_pct = 100.0 * w0_active / w0_total
            w0_inactive_pct = 100.0 * w0_inactive / w0_total
            print(f"window[0] active count: {w0_active}")
            print(f"window[0] inactive count: {w0_inactive}")
            print(f"window[0] active percentage: {w0_active_pct:.4f}%")
            print(f"window[0] inactive percentage: {w0_inactive_pct:.4f}%")
    else:
        _record(checks, 7, "internal consistency checks executable", False, "Missing one or more data arrays")

    _print_h2("8. Sparsity / occupancy statistics")
    if all(x is not None for x in [hb, am, nw]) and hb.shape[0] > 0:
        idxs = sorted(set([0, hb.shape[0] // 2, hb.shape[0] - 1]))
        for idx in idxs:
            active_count = int(np.sum(am[idx]))
            total = int(am[idx].size)
            inactive_count = total - active_count
            active_pct = 100.0 * active_count / total
            hb_min, hb_max = _safe_min_max(hb[idx])
            nw_min, nw_max = _safe_min_max(nw[idx])
            print(f"window[{idx}] active={active_count}, inactive={inactive_count}, active%={active_pct:.4f}, "
                  f"height_bins[min,max]=[{hb_min},{hb_max}], normalized_window[min,max]=[{nw_min},{nw_max}]")

        active_counts = np.sum(am.astype(bool), axis=(1, 2))
        total_cells = am.shape[1] * am.shape[2]
        active_pcts = 100.0 * active_counts / total_cells
        print("dataset-level summary:")
        print(f"mean active count: {active_counts.mean():.4f}")
        print(f"min active count: {active_counts.min()}")
        print(f"max active count: {active_counts.max()}")
        print(f"mean active percentage: {active_pcts.mean():.4f}%")
        _record(checks, 8, "sparsity stats computed", True, f"windows={hb.shape[0]}")
    else:
        _record(checks, 8, "sparsity stats computed", False, "Missing data arrays or zero windows")

    _print_h2("9. Continuous-versus-quantized agreement")
    if all(x is not None for x in [hb, am, nw]):
        active = am.astype(bool)
        quantized = np.floor(nw * 24.0)
        quantized = np.clip(quantized, 0, 23).astype(np.uint8)

        hb_active = hb[active]
        q_active = quantized[active]
        if hb_active.size == 0:
            _record(checks, 9, "quantized agreement on active cells", False, "No active cells found")
        else:
            matches = (hb_active == q_active)
            match_count = int(np.sum(matches))
            mismatch_count = int(np.sum(~matches))
            mismatch_pct = 100.0 * mismatch_count / hb_active.size
            _record(checks, 9, "quantized agreement on active cells", mismatch_count == 0,
                    f"match_count={match_count}, mismatch_count={mismatch_count}, mismatch_pct={mismatch_pct:.6f}%")

            print(f"exact match count: {match_count}")
            print(f"mismatch count: {mismatch_count}")
            print(f"mismatch percentage: {mismatch_pct:.6f}%")

            if mismatch_count > 0:
                mismatch_idx = np.argwhere(active & (hb != quantized))
                print("sample mismatches (window, band, frame, height_bins, recomputed):")
                for row in mismatch_idx[:10]:
                    w, b, f = map(int, row)
                    print((w, b, f, int(hb[w, b, f]), int(quantized[w, b, f])))
    else:
        _record(checks, 9, "quantized agreement on active cells", False, "Missing data arrays")

    _print_h2("10. Visual inspection section")
    if all(x is not None for x in [hb, am, nw]) and hb.shape[0] > 0:
        indices = sorted(set([0, hb.shape[0] // 2, hb.shape[0] - 1]))
        print("Plotting windows:", indices)
        _show_visuals(nw, hb, am, indices)
        _record(checks, 10, "required visual plots rendered", True, f"indices={indices}")
    else:
        _record(checks, 10, "required visual plots rendered", False, "Missing data arrays or no windows")

    _print_h2("11. Manifest alignment checks")
    window_rows = None
    if window_manifest_path is None:
        _record(checks, 11, "window_manifest_path provided", False, "Path is empty")
    elif not window_manifest_path.exists():
        _record(checks, 11, "window_manifest_path exists", False, f"File not found: {window_manifest_path}")
    else:
        try:
            wdf = pd.read_parquet(window_manifest_path)
            window_rows, method = _find_matching_rows(wdf, npz_path)
            context["window_rows"] = window_rows
            _record(checks, 11, "window manifest loaded", True, f"rows={len(wdf)}; match_method={method}; matched_rows={len(window_rows)}")

            if N is not None:
                _record(checks, 11, "manifest row count for selected clip equals N", len(window_rows) == N, f"matched_rows={len(window_rows)}, N={N}")

            if len(window_rows) > 0:
                wr = window_rows.copy()
                if "tensor_index" in wr.columns:
                    wr = wr.sort_values("tensor_index")
                    ti = wr["tensor_index"].to_numpy()
                    if N is not None:
                        _record(checks, 11, "tensor_index runs from 0 to N-1", np.array_equal(ti, np.arange(N)),
                                f"first={ti[:5]}, last={ti[-5:] if len(ti) else []}")
                else:
                    _record(checks, 11, "tensor_index column exists", False, "missing tensor_index")

                if "start_frame" in wr.columns and fs is not None:
                    sf = wr["start_frame"].to_numpy()
                    if N is not None and len(sf) == N:
                        _record(checks, 11, "manifest start_frame matches frame_starts", np.array_equal(sf, fs),
                                f"manifest_first={sf[:5]}, npz_first={fs[:5]}")
                    else:
                        _record(checks, 11, "manifest start_frame matches frame_starts", False,
                                f"length mismatch manifest={len(sf)} npz={len(fs) if fs is not None else 'None'}")
                else:
                    _record(checks, 11, "manifest has start_frame", False, "missing start_frame or frame_starts")

                if "end_frame_exclusive" in wr.columns and "start_frame" in wr.columns:
                    efe_ok = np.array_equal(wr["end_frame_exclusive"].to_numpy(), wr["start_frame"].to_numpy() + 64)
                    _record(checks, 11, "end_frame_exclusive == start_frame + 64", bool(efe_ok), "checked")
                else:
                    _record(checks, 11, "end_frame_exclusive == start_frame + 64", False, "missing columns")

                if "start_sec" in wr.columns and "start_frame" in wr.columns:
                    ss_ok = np.allclose(wr["start_sec"].to_numpy(), wr["start_frame"].to_numpy() * 0.016)
                    _record(checks, 11, "start_sec == start_frame * 0.016", bool(ss_ok), "checked with allclose")
                else:
                    _record(checks, 11, "start_sec == start_frame * 0.016", False, "missing columns")

                if "end_sec" in wr.columns and "end_frame_exclusive" in wr.columns:
                    es_ok = np.allclose(wr["end_sec"].to_numpy(), wr["end_frame_exclusive"].to_numpy() * 0.016)
                    _record(checks, 11, "end_sec == end_frame_exclusive * 0.016", bool(es_ok), "checked with allclose")
                else:
                    _record(checks, 11, "end_sec == end_frame_exclusive * 0.016", False, "missing columns")
            else:
                _record(checks, 11, "selected npz has matching window-manifest rows", False, "no matching rows")
        except Exception as exc:
            _record(checks, 11, "window manifest loaded", False, f"error: {exc}")

    # optional clip manifest checks
    if clip_manifest_path is None or str(clip_manifest_path).strip() == "":
        _record(checks, 11, "clip_manifest optional checks", True, "clip manifest not provided; skipped")
    elif not clip_manifest_path.exists():
        _record(checks, 11, "clip_manifest_path exists", False, f"File not found: {clip_manifest_path}")
    else:
        try:
            cdf = pd.read_parquet(clip_manifest_path)
            clip_rows, clip_method = _find_matching_rows(cdf, npz_path)
            context["clip_rows"] = clip_rows
            _record(checks, 11, "clip manifest loaded", True, f"rows={len(cdf)}; match_method={clip_method}; matched_rows={len(clip_rows)}")

            _record(checks, 11, "corresponding clip row exists", len(clip_rows) >= 1, f"matched_rows={len(clip_rows)}")
            if len(clip_rows) >= 1 and N is not None and "num_windows_total" in clip_rows.columns:
                nws = clip_rows["num_windows_total"].iloc[0]
                _record(checks, 11, "clip num_windows_total == N", int(nws) == int(N), f"num_windows_total={nws}, N={N}")
            elif len(clip_rows) >= 1:
                _record(checks, 11, "clip num_windows_total == N", False, "missing num_windows_total or N")

            if len(clip_rows) >= 1:
                path_cols = [c for c in clip_rows.columns if any(tok in c.lower() for tok in ["tensor", "npz", "path", "file"])]
                if path_cols:
                    path_ok = False
                    npz_abs = _normalize_path_str(Path(npz_path).resolve())
                    npz_name = Path(npz_path).name
                    for col in path_cols:
                        v = _normalize_path_str(str(clip_rows.iloc[0][col]))
                        if v == npz_abs or v.endswith("/" + npz_name) or v == npz_name:
                            path_ok = True
                            break
                    _record(checks, 11, "clip tensor path matches selected file", path_ok, f"checked columns={path_cols}")
                else:
                    _record(checks, 11, "clip tensor path matches selected file", False, "no path-like columns")
        except Exception as exc:
            _record(checks, 11, "clip manifest loaded", False, f"error: {exc}")

    _print_h2("12. Source identity / uniqueness checks")
    wr = context.get("window_rows")
    if wr is None or len(wr) == 0:
        _record(checks, 12, "source identity available", False, "No matched window rows")
    else:
        identity_cols = [
            c for c in wr.columns
            if any(tok in c.lower() for tok in ["source", "relative", "label", "machine", "file", "path"])
        ]
        identity_cols = sorted(set(identity_cols))
        if not identity_cols:
            _record(checks, 12, "identity columns present", False, "No source/path/label/machine columns detected")
        else:
            print("Identity columns detected:", identity_cols)
            summary_rows = []
            for c in identity_cols:
                vals = wr[c].dropna().astype(str).unique().tolist()
                summary_rows.append({
                    "column": c,
                    "unique_count": len(vals),
                    "sample_values": vals[:3],
                })
            identity_df = pd.DataFrame(summary_rows)
            display(identity_df)
            # uniqueness sanity: at least one path/source-like column should be unique per clip match
            path_like = [c for c in identity_cols if any(tok in c.lower() for tok in ["source", "path", "file", "relative"])]
            unique_ok = False
            for c in path_like:
                if wr[c].nunique(dropna=True) == 1:
                    unique_ok = True
                    break
            _record(checks, 12, "selected clip is uniquely identified in manifest slice", unique_ok,
                    f"path_like_columns={path_like}")

    _print_h2("13. Final verdict section")
    checks_df = pd.DataFrame(checks)
    pass_checks = checks_df[checks_df["passed"] == True]
    fail_checks = checks_df[checks_df["passed"] == False]

    print("PASS / READY")
    if len(pass_checks) == 0:
        print("- none")
    else:
        for _, row in pass_checks.iterrows():
            print(f"- [S{row['section']}] {row['check']}")

    print("\nFAIL / PATCH REQUIRED")
    if len(fail_checks) == 0:
        print("- none")
    else:
        for _, row in fail_checks.iterrows():
            print(f"- [S{row['section']}] {row['check']} :: {row['detail']}")

    verdict = "READY_TO_PROCEED" if len(fail_checks) == 0 else "PATCH_REQUIRED"
    print("\nFINAL_VERDICT:", verdict)
    print(verdict)

    context["checks_df"] = checks_df
    context["verdict"] = verdict

    return checks, context


run_button = widgets.Button(description="Run full validation pass", button_style="success")
run_output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="6px"))


def _run_validation(_):
    with run_output:
        run_output.clear_output()
        checks, context = run_full_validation(
            npz_path_text.value,
            window_manifest_path_text.value,
            clip_manifest_path_text.value,
        )
        LAST_RUN.clear()
        LAST_RUN.update(context)


run_button.on_click(_run_validation)
display(run_button, run_output)

## 14. Optional convenience features

Helper cells for compact summaries and ad-hoc per-window inspection.

In [ ]:
# Optional: compact summary dataframe of all checks
if LAST_RUN.get("checks_df") is None:
    print("Run the full validation pass first.")
else:
    display(LAST_RUN["checks_df"])

In [ ]:
# Optional: inspect one window index in detail
window_index = 0  # change as needed

if not LAST_RUN:
    print("Run the full validation pass first.")
else:
    arrays = LAST_RUN.get("arrays", {})
    hb = arrays.get("height_bins")
    am = arrays.get("active_mask")
    nw = arrays.get("normalized_window")

    if hb is None or am is None or nw is None:
        print("Required arrays are missing in LAST_RUN context.")
    else:
        N = hb.shape[0]
        if not (0 <= window_index < N):
            print(f"window_index out of range: {window_index} not in [0, {N-1}]")
        else:
            active_count = int(np.sum(am[window_index]))
            total = int(am[window_index].size)
            print(f"window {window_index}: active={active_count}, inactive={total-active_count}, active%={100*active_count/total:.4f}")
            print(f"height_bins min/max: {hb[window_index].min()} / {hb[window_index].max()}")
            print(f"normalized_window min/max: {nw[window_index].min()} / {nw[window_index].max()}")

            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            im0 = axes[0].imshow(nw[window_index], aspect="auto", origin="lower", cmap="viridis")
            axes[0].set_title(f"normalized_window[{window_index}]")
            plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

            im1 = axes[1].imshow(hb[window_index], aspect="auto", origin="lower", cmap="magma")
            axes[1].set_title(f"height_bins[{window_index}]")
            plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

            im2 = axes[2].imshow(am[window_index].astype(float), aspect="auto", origin="lower", cmap="gray")
            axes[2].set_title(f"active_mask[{window_index}]")
            plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

            plt.tight_layout()
            plt.show()

In [ ]:
# Optional: compare several windows side-by-side
window_indices = [0, 1, 2]  # edit as needed

if not LAST_RUN:
    print("Run the full validation pass first.")
else:
    arrays = LAST_RUN.get("arrays", {})
    hb = arrays.get("height_bins")
    am = arrays.get("active_mask")
    nw = arrays.get("normalized_window")

    if hb is None or am is None or nw is None:
        print("Required arrays are missing in LAST_RUN context.")
    else:
        valid = [i for i in window_indices if 0 <= i < hb.shape[0]]
        if not valid:
            print("No valid indices selected.")
        else:
            fig, axes = plt.subplots(len(valid), 3, figsize=(15, 3.5 * len(valid)), squeeze=False)
            for r, idx in enumerate(valid):
                im0 = axes[r][0].imshow(nw[idx], aspect="auto", origin="lower", cmap="viridis")
                axes[r][0].set_title(f"normalized_window[{idx}]")
                plt.colorbar(im0, ax=axes[r][0], fraction=0.046, pad=0.04)

                im1 = axes[r][1].imshow(hb[idx], aspect="auto", origin="lower", cmap="magma")
                axes[r][1].set_title(f"height_bins[{idx}]")
                plt.colorbar(im1, ax=axes[r][1], fraction=0.046, pad=0.04)

                im2 = axes[r][2].imshow(am[idx].astype(float), aspect="auto", origin="lower", cmap="gray")
                axes[r][2].set_title(f"active_mask[{idx}]")
                plt.colorbar(im2, ax=axes[r][2], fraction=0.046, pad=0.04)

            plt.tight_layout()
            plt.show()